In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

c:\Users\Victus\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [4]:
import os

file_path = r"C:\Users\Victus\Pictures\Tugas Akhir\data Responden\Eytracking Collecting\Rekap Data\Rekap Data ST-1 Part 3 .xlsx"

folder = os.path.dirname(file_path)
if not os.path.exists(file_path):
    candidates = [
        f for f in os.listdir(folder)
        if f.lower().startswith("rekap data st-1 part 3") and f.lower().endswith(".xlsx")
    ]
    if not candidates:
        raise FileNotFoundError(
            f"File not found: {file_path}\nAvailable files:\n{os.listdir(folder)}"
        )
    file_path = os.path.join(folder, candidates[0])

df = pd.read_excel(file_path)
print("✅ File berhasil dibaca!")

print("✅ File berhasil dibaca!")

✅ File berhasil dibaca!
✅ File berhasil dibaca!


In [5]:
import os

folder = r"C:\Users\Victus\Pictures\Tugas Akhir\data Responden\Eytracking Collecting\Rekap Data"

print(os.listdir(folder))

['hasil_eye_tracking ST-1 Part 2.xlsx', 'hasil_eye_tracking ST-1.xlsx', 'hasil_eye_tracking ST-2 Part 2.xlsx', 'hasil_eye_tracking ST-2.xlsx', 'hasil_eye_tracking ST-3 Part 2.xlsx', 'hasil_eye_tracking ST-3.xlsx', 'hasil_eye_tracking ST-4 Part 2.xlsx', 'hasil_eye_tracking ST-4.xlsx', 'hasil_eye_tracking ST-5 Part 2.xlsx', 'hasil_eye_tracking ST-5.xlsx', 'hasil_eye_tracking ST-6 Part 2.xlsx', 'hasil_eye_tracking ST-6.xlsx', 'Rekap Data ST-1 Part 2 .xlsx', 'Rekap Data ST-1 Part 3.xlsx', 'Rekap Data ST-1.xlsx', 'Rekap Data ST-2 Part 2.xlsx', 'Rekap data ST-2.xlsx', 'Rekap Data ST-3 Part 2.xlsx', 'Rekap Data ST-3.xlsx', 'Rekap data ST-4 Part 2.xlsx', 'Rekap data ST-4.xlsx', 'Rekap data ST-5 Part 2.xlsx', 'Rekap data ST-5.xlsx', 'Rekap data ST-6 Part 2.xlsx', 'Rekap data ST-6.xlsx']


In [10]:
import pandas as pd
import numpy as np

# =========================
# 1. LOAD DATA
# =========================
file_path = r"C:\Users\Victus\Pictures\Tugas Akhir\data Responden\Eytracking Collecting\Rekap Data\Rekap data ST-1 Part 3.xlsx"

df = pd.read_excel(file_path)

# =========================
# 2. FUNGSI PER RESPONDEN
# =========================
def process_responden(group):

    group = group.copy()

    # urutkan berdasarkan waktu
    group = group.sort_values(by='Timestamp')

    # =========================
    # HITUNG VELOCITY
    # =========================
    group['Velocity'] = np.sqrt(
        (group['Gaze X'].diff())**2 +
        (group['Gaze Y'].diff())**2
    ) / group['Timestamp'].diff()

    # hapus NaN
    group = group.dropna()

    # =========================
    # THRESHOLD (Median + MAD)
    # =========================
    median_v = group['Velocity'].median()
    mad = np.median(np.abs(group['Velocity'] - median_v))

    k = 3
    threshold = median_v + k * mad

    group['Threshold'] = threshold

    # =========================
    # KLASIFIKASI
    # =========================
    group['Keterangan'] = np.where(
        group['Velocity'] < threshold,
        'Fixation',
        'Saccade'
    )

    # =========================
    # FIXATION EVENT
    # =========================
    fixation_events = (
        (group['Keterangan'] == 'Fixation') &
        (group['Keterangan'].shift(1) != 'Fixation')
    ).sum()

    # =========================
    # DURASI
    # =========================
    duration = group['Timestamp'].max() - group['Timestamp'].min()

    # =========================
    # FIXATION FREQUENCY
    # =========================
    fixation_freq = fixation_events / duration if duration != 0 else 0

    # simpan ke dataframe
    group['Fixation_Event'] = fixation_events
    group['Duration'] = duration
    group['Fixation_Frequency'] = fixation_freq

    return group


# =========================
# 3. APPLY PER RESPONDEN
# =========================
df_result = df.groupby('Responden', group_keys=False).apply(process_responden)

# =========================
# 4. RINGKASAN PER RESPONDEN
# =========================
summary = df_result.groupby('Responden').agg({
    'Threshold': 'first',
    'Fixation_Event': 'first',
    'Fixation_Frequency': 'first'
}).reset_index()

print("\n=== RINGKASAN ===")
print(summary)

# =========================
# 5. SIMPAN HASIL
# =========================
output_path = r"C:\Users\Victus\Pictures\Tugas Akhir\data Responden\Eytracking Collecting\Rekap Data\hasil_eye_tracking.xlsx"

df_result.to_excel(output_path, index=False)

print("\n✅ Selesai! File tersimpan di:")
print(output_path)

KeyError: 'Responden'

In [ ]:
import pandas as pd
import numpy as np

# 1. LOAD DATA
file_path = r"C:\Users\Victus\Pictures\Tugas Akhir\data Responden\Eytracking Collecting\Rekap Data\hasil_eye_tracking ST-1 Part-2.xlsx"
df = pd.read_excel(file_path)

# 2. CEK KOLOM (penting!)
df.columns = df.columns.str.strip()

# 3. AMBIL DATA SAR
sar = df[df['Responden'] == 'SAR']

# 4. HITUNG
median_v = sar['Velocity'].median()
mad = np.median(np.abs(sar['Velocity'] - median_v))
threshold = median_v + 3 * mad

# 5. OUTPUT
print("Median:", median_v)
print("MAD:", mad)
print("Threshold:", threshold)

Median: 0.16142802956265123
MAD: 0.15540611000236565
Threshold: 0.6276463595697481


In [ ]:
import pandas as pd
import numpy as np

# =========================
# 1. LOAD DATA
# =========================
file_path = r"C:\Users\Victus\Pictures\Tugas Akhir\data Responden\Eytracking Collecting\Rekap Data\Rekap Data ST-1 Part 3.xlsx"

df = pd.read_excel(file_path)

print("Kolom dataset:")
print(df.columns.tolist())

# =========================
# 2. FUNGSI PER RESPONDEN
# =========================
def process_responden(group):

    group = group.copy()

    # urutkan berdasarkan waktu
    group = group.sort_values(by='Timestamp')

    # =========================
    # HITUNG VELOCITY
    # =========================
    group['Velocity'] = np.sqrt(
        (group['Gaze X'].diff())**2 +
        (group['Gaze Y'].diff())**2
    ) / group['Timestamp'].diff()

    # hapus NaN
    group = group.dropna()

    # jika data habis setelah dropna
    if len(group) == 0:
        return pd.DataFrame()

    # =========================
    # THRESHOLD (Median + MAD)
    # =========================
    median_v = group['Velocity'].median()

    mad = np.median(
        np.abs(group['Velocity'] - median_v)
    )

    k = 3

    threshold = median_v + (k * mad)

    group['Threshold'] = threshold

    # =========================
    # KLASIFIKASI
    # =========================
    group['Keterangan'] = np.where(
        group['Velocity'] < threshold,
        'Fixation',
        'Saccade'
    )

    # =========================
    # FIXATION EVENT
    # =========================
    fixation_events = (
        (group['Keterangan'] == 'Fixation') &
        (group['Keterangan'].shift(1) != 'Fixation')
    ).sum()

    # =========================
    # DURASI
    # =========================
    duration = (
        group['Timestamp'].max() -
        group['Timestamp'].min()
    )

    # =========================
    # FIXATION FREQUENCY
    # =========================
    fixation_freq = (
        fixation_events / duration
        if duration != 0 else 0
    )

    # =========================
    # SIMPAN HASIL
    # =========================
    group['Fixation_Event'] = fixation_events
    group['Duration'] = duration
    group['Fixation_Frequency'] = fixation_freq

    return group


# =========================
# 3. APPLY PER RESPONDEN
# =========================
df_result = (
    df.groupby('Responden', group_keys=False)
      .apply(process_responden)
      .reset_index(drop=True)
)

# =========================
# DEBUG
# =========================
print("\nKolom setelah apply:")
print(df_result.columns.tolist())

# =========================
# 4. RINGKASAN PER RESPONDEN
# =========================
if 'Responden' not in df_result.columns:

    if 'Responden' not in df_result.columns:
        # coba kembalikan dari index jika Responden disimpan sebagai index
        if df_result.index.name == 'Responden' or 'Responden' in df_result.index.names:
            df_result = df_result.reset_index()
        else:
            # jika tidak, gabungkan kembali dari df asli berdasarkan kolom unik
            df_result = df_result.merge(
                df[['Responden', 'Gaze X', 'Gaze Y', 'Timestamp']],
                on=['Gaze X', 'Gaze Y', 'Timestamp'],
                how='left'
            )

        if 'Responden' not in df_result.columns:
            raise ValueError(
                "Kolom 'Responden' hilang setelah proses apply(). "
                "Periksa nama kolom pada dataset."
            )

summary = (
    df_result[
        [
            'Responden',
            'Threshold',
            'Fixation_Event',
            'Fixation_Frequency'
        ]
    ]
    .drop_duplicates(subset='Responden')
    .reset_index(drop=True)
)

print("\n=== RINGKASAN ===")
print(summary)

# =========================
# 5. SIMPAN HASIL DETAIL
# =========================
output_detail = r"C:\Users\Victus\Pictures\Tugas Akhir\data Responden\Eytracking Collecting\Rekap Data\hasil_eye_tracking_detail.xlsx"

df_result.to_excel(output_detail, index=False)

# =========================
# 6. SIMPAN RINGKASAN
# =========================
output_summary = r"C:\Users\Victus\Pictures\Tugas Akhir\data Responden\Eytracking Collecting\Rekap Data\hasil_eye_tracking_ringkasan.xlsx"

summary.to_excel(output_summary, index=False)

print("\n✅ Selesai!")
print("Detail :", output_detail)
print("Ringkasan :", output_summary)

Kolom dataset:
['Responden', 'Gaze X', 'Gaze Y', 'Timestamp']

Kolom setelah apply:
['Gaze X', 'Gaze Y', 'Timestamp', 'Velocity', 'Threshold', 'Keterangan', 'Fixation_Event', 'Duration', 'Fixation_Frequency']


ValueError: Kolom 'Responden' hilang setelah proses apply(). Periksa nama kolom pada dataset.